# Inference using RF-DETR ONNX model with SAHI sliced inference

RF-DETR Object Detection on drone images (42 MP).
Model format: ONNX (exported from .pt checkpoint).
Slicing: SAHI `get_sliced_prediction` with 896×896 tiles, 25% overlap.


In [ ]:
import os
from google.colab import userdata
# don't need the API key if using fine tuned RF-DETR model downloaded from huggingface
# os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")


In [ ]:
!nvidia-smi


In [ ]:
# rfdetr is still needed for the ONNX export step (Cell 8).
# sahi[roboflow] provides the SAHI slicing framework.
!pip install -q rfdetr==1.5.1 supervision==0.27.0 roboflow
!pip install -q "sahi[roboflow]"
!pip install -q "rfdetr[onnxexport]==1.5.1"


In [ ]:
## set the threshold for inference
THRESHOLD = 0.45

# Slice size — matches model training resolution
SLICE_SIZE = 896


In [ ]:
import os
HOME = os.getcwd()
print(HOME)

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Create uavs2 folder on Drive if it doesn't exist
folders = ["uavs2/"]
for folder in folders:
    path = "/content/drive/MyDrive/" + folder
    if not os.path.exists(path):
        os.mkdir(path)

!mkdir -p {HOME}/datasets
!mkdir -p /content/datasets/savedir/images/
!mkdir -p /content/datasets/test/images/

# Sync images from Drive
!rsync -a "/content/drive/MyDrive/uavs2/images/" "/content/datasets/savedir/images/"
!cp "/content/drive/MyDrive/uavs2/data.yaml" "/content/datasets/data.yaml"
!rsync -a "/content/datasets/savedir/images/" "/content/datasets/test/images/"

!ls /content/datasets/


In [ ]:
# Utility: convert YOLO format labels to COCO JSON.
# Not needed if _annotations.coco.json already exists on Drive.
import os
import json
from PIL import Image

def convert_yolo_to_coco(yolo_images_path, yolo_labels_path, output_coco_path, class_names):
    coco_data = {
        "info": {"year": "2025", "version": "0", "description": "UAVS",
                 "contributor": "", "url": "", "date_created": "2025-01-01T00:00:00+00:00"},
        "licenses": [{"id": 1, "url": "https://creativecommons.org/licenses/by/4.0/", "name": "CC BY 4.0"}],
        "images": [], "annotations": [], "categories": []
    }
    for i, name in enumerate(class_names):
        coco_data["categories"].append({"id": i, "name": name, "supercategory": "none"})
    image_id = 0
    annotation_id = 0
    for img_filename in os.listdir(yolo_images_path):
        if not img_filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        img_path = os.path.join(yolo_images_path, img_filename)
        label_filename = os.path.splitext(img_filename)[0] + '.txt'
        label_path = os.path.join(yolo_labels_path, label_filename)
        with Image.open(img_path) as img:
            width, height = img.size
        coco_data["images"].append({"id": image_id, "file_name": img_filename,
                                     "width": width, "height": height})
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)
                    abs_x_center = x_center * width
                    abs_y_center = y_center * height
                    abs_bbox_width = bbox_width * width
                    abs_bbox_height = bbox_height * height
                    x_min = abs_x_center - (abs_bbox_width / 2)
                    y_min = abs_y_center - (abs_bbox_height / 2)
                    coco_data["annotations"].append({
                        "id": annotation_id, "image_id": image_id,
                        "category_id": int(class_id),
                        "bbox": [x_min, y_min, abs_bbox_width, abs_bbox_height],
                        "area": abs_bbox_width * abs_bbox_height, "iscrowd": 0
                    })
                    annotation_id += 1
        image_id += 1
    with open(output_coco_path, 'w') as f:
        json.dump(coco_data, f, indent=4)


In [ ]:
# Copy ground-truth annotations from Drive into the Colab instance
!cp "/content/drive/MyDrive/uavs2/_annotations.coco.json" "/content/datasets/test/_annotations.coco.json"


In [ ]:
# GPU memory cleanup utility — used after the ONNX export to free memory
# before loading the inference session.
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):
    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()
    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")


In [ ]:
# fix pillow import error
!pip install -q "Pillow==10.4.0"

In [ ]:
## Export .pt checkpoint → ONNX, and read class names from annotations.
## This cell only needs to run once; if the .onnx already exists on Drive it is skipped.

import os, json, gc, torch
from rfdetr import RFDETRLarge

PT_WEIGHTS       = "/content/drive/MyDrive/uavs2/checkpoint_best_total.pth"
ONNX_MODEL_PATH  = "/content/drive/MyDrive/uavs2/rf_detr_large.onnx"
ANNOTATIONS_PATH = "/content/datasets/test/_annotations.coco.json"

# Read class names from the COCO annotations file
with open(ANNOTATIONS_PATH, "r") as f:
    coco_meta = json.load(f)

CLASS_NAMES      = [cat["name"] for cat in sorted(coco_meta["categories"], key=lambda c: c["id"])]
CATEGORY_MAPPING = {str(i): name for i, name in enumerate(CLASS_NAMES)}
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
print(f"CATEGORY_MAPPING: {CATEGORY_MAPPING}")

# Export to ONNX (skip if already done)
if os.path.exists(ONNX_MODEL_PATH):
    print(f"\nONNX file already exists, skipping export:\n  {ONNX_MODEL_PATH}")
else:
    print(f"\nExporting {PT_WEIGHTS} → ONNX ...")
    export_model = RFDETRLarge(
        pretrain_weights=PT_WEIGHTS,
        resolution=896,
        num_classes=len(CLASS_NAMES),
    )
    export_model.export()
    import shutil
    shutil.copy("output/inference_model.onnx", ONNX_MODEL_PATH)
    del export_model
    cleanup_gpu_memory(verbose=True)
    print(f"Exported and saved to {ONNX_MODEL_PATH}")


In [ ]:
## Load the ONNX model into a custom SAHI DetectionModel subclass.
## This is the model that all inference cells below will use.

import torch
import numpy as np
import supervision as sv
import onnxruntime as ort
from PIL import Image
from sahi.models.base import DetectionModel
from sahi.prediction import ObjectPrediction
from sahi.predict import get_sliced_prediction


class RFDetrOnnxDetectionModel(DetectionModel):
    """SAHI-compatible wrapper for RF-DETR exported as ONNX.

    Output tensors from rfdetr 1.5.1 ONNX export:
      dets:   [1, 300, 4]  normalised cx,cy,w,h
      labels: [1, 300, 1]  raw logits (sigmoid → confidence score)
    """

    def load_model(self):
        providers = (
            ["CUDAExecutionProvider", "CPUExecutionProvider"]
            if torch.cuda.is_available()
            else ["CPUExecutionProvider"]
        )
        self.session = ort.InferenceSession(self.model_path, providers=providers)
        self.input_name  = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        print(f"ONNX inputs : {self.input_name}")
        print(f"ONNX outputs: {self.output_names}")
        self.model = self.session

    def perform_inference(self, image: np.ndarray):
        h, w = image.shape[:2]
        tile = Image.fromarray(image).resize(
            (self.image_size, self.image_size), Image.BILINEAR
        )
        tile = np.array(tile, dtype=np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        tile = (tile - mean) / std
        tile = tile.transpose(2, 0, 1)[None]
        outputs = self.session.run(self.output_names, {self.input_name: tile})
        self._original_predictions = outputs
        self._tile_hw = (h, w)

    def convert_original_predictions(self, shift_amount=None, full_shape=None):
        if shift_amount is None:
            shift_amount = [0, 0]

        dets   = self._original_predictions[0][0]   # [300, 4]  cx,cy,w,h normalised
        labels = self._original_predictions[1][0]   # [300, 1]  raw logits
        tile_h, tile_w = self._tile_hw

        # Confidence = sigmoid(logit)
        scores = 1.0 / (1.0 + np.exp(-labels[:, 0]))   # [300]

        object_prediction_list = []
        for box, score in zip(dets, scores):
            if score < self.confidence_threshold:
                continue
            cx, cy, bw, bh = box
            x1 = (cx - bw / 2) * tile_w
            y1 = (cy - bh / 2) * tile_h
            x2 = (cx + bw / 2) * tile_w
            y2 = (cy + bh / 2) * tile_h
            x1, y1 = max(0.0, x1), max(0.0, y1)
            x2, y2 = min(float(tile_w), x2), min(float(tile_h), y2)
            if x2 <= x1 or y2 <= y1:
                continue
            category_name = self.category_mapping.get("0", "object")
            object_prediction_list.append(
                ObjectPrediction(
                    bbox=[x1, y1, x2, y2],
                    score=float(score),
                    category_id=0,
                    category_name=category_name,
                    shift_amount=shift_amount,
                    full_shape=full_shape,
                )
            )
        self._object_prediction_list_per_image = [object_prediction_list]


def sahi_result_to_sv_detections(result) -> sv.Detections:
    """Convert a SAHI PredictionResult → supervision.Detections."""
    preds = result.object_prediction_list
    if not preds:
        return sv.Detections.empty()
    xyxy       = np.array([[p.bbox.minx, p.bbox.miny,
                             p.bbox.maxx, p.bbox.maxy] for p in preds], dtype=np.float32)
    confidence = np.array([p.score.value for p in preds], dtype=np.float32)
    class_ids  = np.array([p.category.id  for p in preds], dtype=int)
    return sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_ids)


# Instantiate
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Loading ONNX model on {DEVICE} ...")

detection_model = RFDetrOnnxDetectionModel(
    model_path=ONNX_MODEL_PATH,
    confidence_threshold=THRESHOLD,
    category_mapping=CATEGORY_MAPPING,
    image_size=896,
    device=DEVICE,
)
detection_model.load_model()
print("Model loaded.\n")


In [ ]:
import supervision as sv

dataset = "/content/datasets/"

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset}/test/images",
    annotations_path=f"{dataset}/test/_annotations.coco.json",
)
print(f"Dataset loaded: {len(ds)} images, classes: {ds.classes}")


In [ ]:
# Install GPU version FIRST before any imports
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q "onnxruntime-gpu==1.19.2"

In [ ]:
# Then restart runtime and run this in a fresh cell
import onnxruntime as ort
print("Available providers:", ort.get_available_providers())

sess = ort.InferenceSession(
    ONNX_MODEL_PATH,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
print("Active provider:", sess.get_providers()[0])

In [ ]:
# Check what's available
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Check onnxruntime build
import onnxruntime as ort
print("ORT version:", ort.__version__)
print("Available providers:", ort.get_available_providers())

In [ ]:
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision
from PIL import Image

targets     = []
predictions = []

for path, image, annotations in tqdm(ds, desc="mAP evaluation"):
    image_pil = Image.open(path)

    result = get_sliced_prediction(
        image_pil, detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=0.25, overlap_width_ratio=0.25,
        verbose=0,
    )
    detections = sahi_result_to_sv_detections(result)

    if detections.class_id is not None:
        detections.class_id = np.clip(detections.class_id, 0, len(CLASS_NAMES) - 1)

    targets.append(annotations)
    predictions.append(detections)

map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)

print(type(targets[0]))
print(type(predictions[0]))


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import torch
import torchvision
import os

def match_detections(ground_truth, predictions, iou_threshold=0.5):
    if len(ground_truth) == 0 and len(predictions) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=int)
    elif len(ground_truth) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.arange(len(predictions), dtype=int)
    elif len(predictions) == 0:
        return np.arange(len(ground_truth), dtype=int), np.array([], dtype=int), np.array([], dtype=int)

    ground_truth_boxes = torch.from_numpy(ground_truth.xyxy)
    predictions_boxes  = torch.from_numpy(predictions.xyxy)
    iou_matrix = torchvision.ops.box_iou(ground_truth_boxes, predictions_boxes)
    best_iou_for_gt, best_pred_indices_for_gt = iou_matrix.max(dim=1)

    matched_ground_truth_indices = []
    matched_prediction_indices   = []
    prediction_matched = [False] * len(predictions)

    for i in range(len(ground_truth)):
        best_iou        = best_iou_for_gt[i].item()
        best_pred_index = best_pred_indices_for_gt[i].item()
        if best_iou >= iou_threshold and not prediction_matched[best_pred_index]:
            matched_ground_truth_indices.append(i)
            matched_prediction_indices.append(best_pred_index)
            prediction_matched[best_pred_index] = True

    false_positive_indices = [j for j in range(len(predictions)) if not prediction_matched[j]]
    return (
        np.array(matched_ground_truth_indices, dtype=int),
        np.array(matched_prediction_indices,   dtype=int),
        np.array(false_positive_indices,        dtype=int),
    )

true_labels      = []
predicted_labels = []

for gt_detections, pred_detections in zip(targets, predictions):
    if gt_detections is None or pred_detections is None:
        continue
    matched_gt, matched_pred, fp_indices = match_detections(gt_detections, pred_detections)
    for gt_idx, pred_idx in zip(matched_gt, matched_pred):
        true_labels.append(gt_detections.class_id[gt_idx])
        predicted_labels.append(pred_detections.class_id[pred_idx])
    for fp_idx in fp_indices:
        true_labels.append(-1)
        predicted_labels.append(pred_detections.class_id[fp_idx])
    all_gt_indices = set(range(len(gt_detections)))
    fn_indices = list(all_gt_indices - set(matched_gt))
    for fn_idx in fn_indices:
        true_labels.append(gt_detections.class_id[fn_idx])
        predicted_labels.append(-1)

unique_labels  = sorted(list(set(true_labels + predicted_labels)))
label_to_index = {label: i for i, label in enumerate(unique_labels)}
true_indices      = [label_to_index[l] for l in true_labels]
predicted_indices = [label_to_index[l] for l in predicted_labels]
cm = confusion_matrix(true_indices, predicted_indices, labels=list(label_to_index.values()))

cm_labels   = [None] * len(unique_labels)
class_names = ds.classes
for label, index in label_to_index.items():
    if label == -1:
        cm_labels[index] = 'No Object'
    else:
        try:
            cm_labels[index] = class_names[label]
        except IndexError:
            cm_labels[index] = f"Unknown Class {label}"

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]),
       xticklabels=cm_labels, yticklabels=cm_labels,
       title='Confusion Matrix', ylabel='True label', xlabel='Predicted label')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
fmt    = 'd'
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
fig.tight_layout()

os.makedirs('/content/datasets/output', exist_ok=True)
output_path = '/content/datasets/output/confusion_matrix.png'
plt.savefig(output_path)
plt.show()


In [ ]:
import shutil

source      = "/content/datasets/output/confusion_matrix.png"
destination = "/content/drive/MyDrive/uavs2/output/sliced_threshold_45_confusion_matrix.png"

try:
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    shutil.copy2(source, destination)
    print(f"Successfully copied {source} to {destination}")
except FileNotFoundError:
    print(f"Error: Source file {source} not found")
except Exception as e:
    print(f"Error copying file: {e}")


In [ ]:
## Single-image inference and visualisation — ds[51]

from PIL import Image

path, image, annotations = ds[51]
image = Image.open(path)

result = get_sliced_prediction(
    image, detection_model,
    slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
    overlap_height_ratio=0.25, overlap_width_ratio=0.25,
    verbose=0,
)
detections = sahi_result_to_sv_detections(result)
if detections.class_id is not None:
    detections.class_id = np.clip(detections.class_id, 0, len(CLASS_NAMES) - 1)

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness  = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

bbox_annotator  = sv.BoxAnnotator(color=color, thickness=thickness)
label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.BLACK, text_scale=text_scale)

annotations_labels = [f"{ds.classes[class_id]}" for class_id in annotations.class_id]
detections_labels  = [
    f"{ds.classes[cid]} {conf:.2f}"
    for cid, conf in zip(detections.class_id, detections.confidence)
]

annotation_image = image.copy()
annotation_image = bbox_annotator.annotate(annotation_image, annotations)
annotation_image = label_annotator.annotate(annotation_image, annotations, annotations_labels)

detections_image = image.copy()
detections_image = bbox_annotator.annotate(detections_image, detections)
detections_image = label_annotator.annotate(detections_image, detections, detections_labels)

sv.plot_images_grid(
    images=[annotation_image, detections_image],
    grid_size=(1, 2),
    titles=["Ground Truth", "ONNX + SAHI Detections"]
)


In [ ]:
## Run inference over all test images.
## Saves annotated images and a COCO-format predictions JSON.

import os
import json
import shutil
from PIL import Image
from tqdm import tqdm
from datetime import datetime

TEST_IMAGES_DIR      = "/content/datasets/test/images"
OUTPUT_DIR           = "/content/datasets/output/inference"
ANNOTATED_IMAGES_DIR = os.path.join(OUTPUT_DIR, "annotated_images")
COCO_ANNOTATIONS_PATH = os.path.join(OUTPUT_DIR, "predictions_coco.json")

os.makedirs(ANNOTATED_IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
test_images = sorted([
    f for f in os.listdir(TEST_IMAGES_DIR)
    if f.lower().endswith(image_extensions)
])

print(f"Found {len(test_images)} images in test set")
print(f"Output will be saved to: {OUTPUT_DIR}")

coco_data = {
    "info": {
        "description": "RF-DETR ONNX + SAHI Inference Results",
        "date_created": datetime.now().isoformat(),
        "model": "RFDETRLarge ONNX",
        "threshold": THRESHOLD,
        "slice_size": SLICE_SIZE,
    },
    "images": [],
    "annotations": [],
    "categories": [{"id": i, "name": name, "supercategory": "none"}
                   for i, name in enumerate(CLASS_NAMES)],
}

annotation_id = 0
image_id      = 0

color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

for img_filename in tqdm(test_images, desc="Processing images"):
    img_path  = os.path.join(TEST_IMAGES_DIR, img_filename)
    image_pil = Image.open(img_path).convert("RGB")
    width, height = image_pil.size

    result = get_sliced_prediction(
        image_pil, detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=0.25, overlap_width_ratio=0.25,
        verbose=0,
    )
    detections = sahi_result_to_sv_detections(result)
    if detections.class_id is not None:
        detections.class_id = np.clip(detections.class_id, 0, len(CLASS_NAMES) - 1)

    coco_data["images"].append({
        "id": image_id, "file_name": img_filename,
        "width": width, "height": height,
    })

    if len(detections) > 0:
        xyxy       = detections.xyxy
        confidence = detections.confidence
        class_ids  = detections.class_id
        for i in range(len(detections)):
            x1, y1, x2, y2 = xyxy[i]
            bw = x2 - x1
            bh = y2 - y1
            coco_data["annotations"].append({
                "id":          annotation_id,
                "image_id":    image_id,
                "category_id": int(class_ids[i]) if class_ids is not None else 0,
                "bbox":        [float(x1), float(y1), float(bw), float(bh)],
                "area":        float(bw * bh),
                "score":       float(confidence[i]) if confidence is not None else 1.0,
                "iscrowd":     0,
            })
            annotation_id += 1

    # Annotated image
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=(width, height))
    thickness  = sv.calculate_optimal_line_thickness(resolution_wh=(width, height))
    bbox_annotator  = sv.BoxAnnotator(color=color, thickness=thickness)
    label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.BLACK,
                                         text_scale=text_scale, smart_position=True)
    annotated_image = image_pil.copy()
    if len(detections) > 0:
        labels = [
            f"object {conf:.2f}" if conf else "object"
            for conf in (detections.confidence if detections.confidence is not None
                         else [None] * len(detections))
        ]
        annotated_image = bbox_annotator.annotate(annotated_image, detections)
        annotated_image = label_annotator.annotate(annotated_image, detections, labels)
    annotated_image.save(os.path.join(ANNOTATED_IMAGES_DIR, img_filename))
    image_id += 1

with open(COCO_ANNOTATIONS_PATH, 'w') as f:
    json.dump(coco_data, f, indent=2)

print(f"\n✅ Inference complete!")
print(f"   - Processed {len(test_images)} images")
print(f"   - Found {annotation_id} total detections")
print(f"   - Annotated images saved to: {ANNOTATED_IMAGES_DIR}")
print(f"   - COCO predictions saved to: {COCO_ANNOTATIONS_PATH}")

summary_path = os.path.join(OUTPUT_DIR, "inference_summary.txt")
with open(summary_path, 'w') as f:
    f.write(f"Inference Summary\n")
    f.write(f"================\n")
    f.write(f"Date: {datetime.now().isoformat()}\n")
    f.write(f"Model: RFDETRLarge ONNX via SAHI\n")
    f.write(f"ONNX path: {ONNX_MODEL_PATH}\n")
    f.write(f"Confidence Threshold: {THRESHOLD}\n")
    f.write(f"Slice Size: {SLICE_SIZE}\n")
    f.write(f"Total Images Processed: {len(test_images)}\n")
    f.write(f"Total Detections: {annotation_id}\n")
    f.write(f"Output Directory: {OUTPUT_DIR}\n")
print(f"   - Summary saved to: {summary_path}")

# Copy to Drive
try:
    DRIVE_OUTPUT = "/content/drive/MyDrive/uavs2/inference_results_onnx"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f"   - Results copied to Google Drive: {DRIVE_OUTPUT}")
except Exception as e:
    print(f"   - Note: Could not copy to Google Drive: {e}")


In [ ]:
## Quick sanity check on the COCO predictions JSON

import json
from collections import Counter

with open(COCO_ANNOTATIONS_PATH) as f:
    coco_out = json.load(f)

total    = len(coco_out["annotations"])
n_images = len(coco_out["images"])
print(f"Total detections:          {total}")
print(f"Images:                    {n_images}")
print(f"Mean detections per image: {total/n_images:.1f}")

counts = Counter(a["image_id"] for a in coco_out["annotations"])
print("\nTop 5 images by detection count:")
for img_id, count in counts.most_common(5):
    fname = next(i["file_name"] for i in coco_out["images"] if i["id"] == img_id)
    print(f"  {fname}: {count} detections")


In [ ]:
from IPython.display import display
from PIL import Image

sample_img_path = os.path.join(ANNOTATED_IMAGES_DIR, test_images[0])
if os.path.exists(sample_img_path):
    display(Image.open(sample_img_path))
    print(f"\n📸 Preview: {test_images[0]}")
